In [4]:
import os
import pandas as pd
import numpy as np


#### Isolate the needed rooms from all of the cleaned schedules in to mapped csvs

In [1]:


# Directory containing the CSV files
dir_path = '/workspaces/CUBES/exp/jack/paper/thermostat_experiment/data/cleaned_schedules'
output_dir = '/workspaces/CUBES/exp/jack/paper/thermostat_experiment/data/mapped_schedules'

# Ensure output directory exists
os.makedirs(output_dir, exist_ok=True)

# List all files in the directory
files = os.listdir(dir_path)

# Filter to get only CSV files
csv_files = [file for file in files if file.endswith('.csv')]

# Create the mapping dictionary (with simple names)
mapping = {
    'Hall(Down stairs)': 'hall_downstairs',
    'Hall(Upstairs)': 'hall_upstairs',
    'Front Room(Down stairs)': 'front_room',
    'Kitchen(Down stairs)': 'kitchen',
    'Backroom(Down stairs)': 'backroom',
    'Bedroom 1(Upstairs)': 'bedroom_1',
    'Bedroom 2(Upstairs)': 'bedroom_2',
    'Bedroom 3(Upstairs)': 'bedroom_3',
    'Bathroom 1(Upstairs)': 'bathroom',
    # Add more mappings if needed for other columns
}

# List of target names to keep
target_names = [
    "hall_downstairs", "front_room", "kitchen", "backroom",
    "bedroom_3", "bedroom_1", "hall_upstairs", "bathroom", "bedroom_2"
]

# Loop through each CSV file, apply the mapping, and save the result
for csv_file in csv_files:
    file_path = os.path.join(dir_path, csv_file)

    # Read the CSV file
    df = pd.read_csv(file_path)

    # Strip the 'H_XX_' prefix from the columns before mapping
    df.columns = df.columns.str.replace(r'H\d{2}_', '', regex=True)

    # Rename columns based on the mapping dictionary
    df_renamed = df.rename(columns=mapping)

    # Filter the DataFrame to keep only columns that match the target names
    target_columns = [col for col in target_names if col in df_renamed.columns]
    df_filtered = df_renamed[target_columns]

    # Save the modified DataFrame as a new CSV
    output_file_path = os.path.join(output_dir, f"mapped_{csv_file}")
    df_filtered.to_csv(output_file_path, index=False)


<jemalloc>: MADV_DONTNEED does not work (memset will be used instead)
<jemalloc>: (This is the expected behaviour if you are running under QEMU)


#### Create room csv's from these mapped csvs

In [2]:

# Directory containing the mapped CSV files
mapped_dir = '/workspaces/CUBES/exp/jack/paper/thermostat_experiment/data/mapped_schedules'
output_dir = '/workspaces/CUBES/exp/jack/paper/thermostat_experiment/data/room_schedules'

# Ensure output directory exists
os.makedirs(output_dir, exist_ok=True)

# List all mapped CSV files
files = os.listdir(mapped_dir)
csv_files = [file for file in files if file.endswith('.csv')]

# List of target room columns
target_names = [
    "hall_downstairs", "front_room", "kitchen", "backroom",
    "bedroom_3", "bedroom_1", "hall_upstairs", "bathroom", "bedroom_2"
]

# Dictionary to store data for each room
room_data = {room: [] for room in target_names}  # Store dataframes for each room

# Loop through each mapped CSV file and append data to corresponding room
for csv_file in csv_files:
    file_path = os.path.join(mapped_dir, csv_file)

    # Extract house_id from the file name (or use csv_file if house_id isn't part of the filename)
    house_id = os.path.splitext(csv_file)[0]  # Use file name (without extension) as house_id

    # Read the CSV file
    df = pd.read_csv(file_path)

    # For each room, append the relevant data (if available in the current file)
    for room in target_names:
        if room in df.columns:
            # Create a DataFrame for this room's data
            room_df = pd.DataFrame({
                house_id: df[room]
            })

            # Append this DataFrame to the room_data list for the specific room
            room_data[room].append(room_df)

# Concatenate and save all data for each room into separate CSV files
for room, data_list in room_data.items():
    if data_list:  # Ensure there's data for the room
        # Concatenate the data for all house_ids (side by side, column-wise)
        combined_df = pd.concat(data_list, axis=1)

        # Save the combined data into a single CSV file for this room
        output_file_path = os.path.join(output_dir, f"{room}_schedule.csv")
        combined_df.to_csv(output_file_path, index=False)


#### Generate the synthetic data


In [6]:

# Directory containing the room schedules
room_schedule_dir = '/workspaces/CUBES/exp/jack/paper/thermostat_experiment/data/room_schedules'
synthetic_schedule_dir = '/workspaces/CUBES/exp/jack/paper/thermostat_experiment/data/synthetic_schedules'

# Ensure synthetic schedule directory exists
os.makedirs(synthetic_schedule_dir, exist_ok=True)

# List of target room columns (these names will remain in the synthetic schedules)
target_names = [
    "hall_downstairs", "front_room", "kitchen", "backroom",
    "bedroom_3", "bedroom_1", "hall_upstairs", "bathroom", "bedroom_2"
]

# Number of synthetic schedules to generate
num_synthetic_schedules = 20  # rep_0 to rep_19

# Loop to generate each synthetic schedule
for rep in range(num_synthetic_schedules):
    # Dictionary to store synthetic data for all rooms in this synthetic schedule
    synthetic_data = {}

    # Loop through each room schedule
    for room in target_names:
        room_schedule_path = os.path.join(room_schedule_dir, f"{room}_schedule.csv")

        # Read the room schedule
        room_df = pd.read_csv(room_schedule_path)

        # Randomly pick one of the columns (a house_id's data) without changing the room name
        random_column = np.random.choice(room_df.columns)

        # Store the selected column's data under the room name (ignoring the house_id)
        synthetic_data[room] = room_df[random_column]

    # Create a DataFrame from the synthetic data (keeping room names as columns)
    synthetic_schedule_df = pd.DataFrame(synthetic_data)

    # Clip the values in the DataFrame to be between 0 and 1
    synthetic_schedule_df = synthetic_schedule_df.clip(lower=0, upper=1)

    # Save the synthetic schedule as a CSV with filenames rep_0.csv to rep_19.csv
    synthetic_schedule_path = os.path.join(synthetic_schedule_dir, f"rep_{rep}.csv")
    synthetic_schedule_df.to_csv(synthetic_schedule_path, index=False)


In [7]:
utc_df = pd.read_csv("/workspaces/CUBES/exp/jack/paper/thermostat_experiment/data/cleaned_schedules/H01_cleaned_2013.csv")
utc_col = utc_df["UTC_Time"]

#### Create heating patterns from synthetic occupancy data

In [12]:

# Directory containing the room schedules and synthetic schedules
synthetic_schedule_dir = '/workspaces/CUBES/exp/jack/paper/thermostat_experiment/data/synthetic_schedules/occupancy'
heating_schedule_dir = '/workspaces/CUBES/exp/jack/paper/thermostat_experiment/data/synthetic_schedules/heating'

# Ensure heating schedule directory exists
os.makedirs(heating_schedule_dir, exist_ok=True)

# Load the UTC_Time column from the existing CSV file
utc_df = pd.read_csv("/workspaces/CUBES/exp/jack/paper/thermostat_experiment/data/cleaned_schedules/H01_cleaned_2013.csv")
utc_col = utc_df["UTC_Time"]

# Rooms to apply the heating rule to
rooms = ['backroom', 'bathroom', 'front_room', 'hall_downstairs', 'bedroom_2', 'kitchen', 'bedroom_1', 'bedroom_3', 'hall_upstairs']

# Define the heating rule function
def apply_heating_rule(series, time_mask):
    heating = np.zeros_like(series)  # Initial heating schedule, all off (0)
    for i in range(5, len(series)):  # Start from 5th element due to 5-minute window
        if time_mask[i]:  # If time is between 23:00 and 07:00, heating is off
            heating[i] = 0
        elif series[i] > 0:  # If current value is greater than 0, heating is on for 5 mins
            heating[i-4:i+1] = 1
        elif np.all(series[i-4:i] == 0):  # If no activity in the last 5 mins, heating off
            heating[i] = 0
    return heating

# Loop through each synthetic schedule (rep_0 to rep_19)
for rep in range(20):
    # Read the synthetic schedule
    synthetic_schedule_path = os.path.join(synthetic_schedule_dir, f"rep_{rep}.csv")
    df = pd.read_csv(synthetic_schedule_path)

    # Check if the UTC column length matches the synthetic schedule length
    if len(df) != len(utc_col):
        raise ValueError(f"Mismatch in lengths: Synthetic schedule for rep_{rep} has {len(df)} rows, but UTC_Time column has {len(utc_col)} rows.")

    # Add the UTC_Time column to the synthetic schedule
    df_copy = df.copy()
    df_copy['UTC_Time'] = pd.to_datetime(utc_col)

    # Create a boolean mask for the time between 23:00 and 07:00
    time_mask = (df_copy['UTC_Time'].dt.hour >= 23) | (df_copy['UTC_Time'].dt.hour < 7)

    # Apply the heating rule to each room in the DataFrame
    for room in rooms:
        df_copy[room] = apply_heating_rule(df[room].values, time_mask)

    # Save the heating schedule as a CSV
    heating_schedule_path = os.path.join(heating_schedule_dir, f"rep_{rep}.csv")
    df_copy.to_csv(heating_schedule_path, index=False)
